In [0]:
# Cria (se não existir) o schema e o volume onde o dado da validação vai morar.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.credit_reject")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.credit_reject.data")
BASE = "/Volumes/workspace/credit_reject/data"
print("Volume pronto em:", BASE)

Volume pronto em: /Volumes/workspace/credit_reject/data


In [0]:
%pip install kaggle

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import subprocess, os
target = f"{BASE}"
# baixa o arquivo específico de rejeitados do dataset
cmd = [
    "kaggle", "datasets", "download",
    "-d", "wordsforthewise/lending-club",
    "-f", "rejected_2007_to_2018Q4.csv.gz",
    "-p", target
]
print("Baixando... (pode levar alguns minutos no Free Edition)")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
print(res.stderr)
# confirma o arquivo no Volume
print(os.listdir(target))

Baixando... (pode levar alguns minutos no Free Edition)
Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
License(s): CC0-1.0
rejected_2007_to_2018Q4.csv.gz: Skipping, found more recently modified local copy (use --force to force download)


['rejected_2007_to_2018Q4.csv.gz']


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

BASE = "/Volumes/workspace/credit_reject/data"
RAW_GZ = f"{BASE}/rejected_2007_to_2018Q4.csv.gz"
PROC = f"{BASE}/processed/rejected.parquet"

CORRUPT = "_corrupt_record"
SRC = ["Amount Requested","Application Date","Loan Title","Risk_Score",
       "Debt-To-Income Ratio","Zip Code","State","Employment Length","Policy Code"]
schema = StructType([StructField(c, StringType(), True) for c in SRC]
                    + [StructField(CORRUPT, StringType(), True)])

df_raw = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", CORRUPT)
    .schema(schema)
    .csv(RAW_GZ))

print("Colunas lidas:", df_raw.columns)

Colunas lidas: ['Amount Requested', 'Application Date', 'Loan Title', 'Risk_Score', 'Debt-To-Income Ratio', 'Zip Code', 'State', 'Employment Length', 'Policy Code', '_corrupt_record']


In [0]:
n = df_raw.count()
print(f"[MEDIDO] Linhas: {n:,}")

n_corrupt = df_raw.filter(F.col(CORRUPT).isNotNull()).count()
print(f"[AUDITORIA] Linhas corrompidas apos parsing robusto: {n_corrupt:,}")
print("OK - zero corrompidas" if n_corrupt == 0 else "ATENCAO - ha corrompidas")

df = df_raw.drop(CORRUPT)

[MEDIDO] Linhas: 27,648,741
[AUDITORIA] Linhas corrompidas apos parsing robusto: 0
OK - zero corrompidas


In [0]:
RENAME = {
    "Amount Requested":"amount_requested", "Application Date":"application_date",
    "Loan Title":"loan_title", "Risk_Score":"risk_score",
    "Debt-To-Income Ratio":"dti_raw", "Zip Code":"zip3", "State":"state",
    "Employment Length":"emp_length_raw", "Policy Code":"policy_code",
}
for s, d in RENAME.items():
    if s in df.columns:
        df = df.withColumnRenamed(s, d)

df = df.withColumn("amount_requested", F.col("amount_requested").cast("double"))
df = df.withColumn("risk_score", F.col("risk_score").cast("double"))
df = df.withColumn("application_date", F.to_date("application_date", "yyyy-MM-dd"))
df = df.withColumn("app_year", F.year("application_date"))
df = df.withColumn("dti", F.regexp_replace(F.col("dti_raw"), "%", "").cast("double"))
print("Colunas apos normalizacao:", df.columns)

Colunas apos normalizacao: ['amount_requested', 'application_date', 'loan_title', 'risk_score', 'dti_raw', 'zip3', 'state', 'emp_length_raw', 'policy_code', 'app_year', 'dti']


In [0]:
# Esta e a operacao que quebrava no Windows por winutils. No Databricks (Linux) roda nativa.
(df.write
   .mode("overwrite")
   .partitionBy("app_year")
   .parquet(PROC))
print(f"[PROVA] Parquet particionado escrito via Spark NATIVO em: {PROC}")

df_back = spark.read.parquet(PROC)
print(f"[VERIFICA] Linhas lidas de volta: {df_back.count():,}")

[PROVA] Parquet particionado escrito via Spark NATIVO em: /Volumes/workspace/credit_reject/data/processed/rejected.parquet
[VERIFICA] Linhas lidas de volta: 27,648,741


In [0]:
total = df_back.count()
print("=== risk_score por ano ===")
(df_back.groupBy("app_year")
   .agg(F.count("*").alias("n"),
        F.count("risk_score").alias("n_risk"),
        F.round(100.0*F.count("risk_score")/F.count("*"), 2).alias("pct_presente"))
   .orderBy("app_year").show(50, truncate=False))

overall = df_back.select(
    F.round(100.0*F.count("risk_score")/F.count("*"), 2)).collect()[0][0]
print(f"[HEADLINE] risk_score presente em {overall}% do total "
      f"(=> {round(100-overall,2)}% ausente)")

=== risk_score por ano ===
+--------+-------+-------+------------+
|app_year|n      |n_risk |pct_presente|
+--------+-------+-------+------------+
|2007    |5274   |5170   |98.03       |
|2008    |25596  |23090  |90.21       |
|2009    |56991  |50765  |89.08       |
|2010    |112561 |104584 |92.91       |
|2011    |217792 |215014 |98.72       |
|2012    |337277 |332939 |98.71       |
|2013    |760942 |737120 |96.87       |
|2014    |1933700|1679177|86.84       |
|2015    |2859379|510451 |17.85       |
|2016    |4769874|1018297|21.35       |
|2017    |7072573|3826324|54.1        |
|2018    |9496782|648180 |6.83        |
+--------+-------+-------+------------+

[HEADLINE] risk_score presente em 33.1% do total (=> 66.9% ausente)


In [0]:
print("=== Top 10 estados (recusados) ===")
(df_back.groupBy("state").count().orderBy(F.desc("count")).show(10, truncate=False))
print("Validacao concluida: Spark nativo leu o gzip e escreveu Parquet particionado no Databricks.")

=== Top 10 estados (recusados) ===
+-----+-------+
|state|count  |
+-----+-------+
|CA   |3242169|
|TX   |2495511|
|FL   |2167584|
|NY   |1991179|
|GA   |1083614|
|PA   |1047694|
|OH   |1011312|
|IL   |1001046|
|NC   |863860 |
|NJ   |853305 |
+-----+-------+
only showing top 10 rows
Validacao concluida: Spark nativo leu o gzip e escreveu Parquet particionado no Databricks.
